# Gold Layer - Analytics

Creates business-oriented analytical datasets from the Silver layer.

## What we do

- Calculate daily sales metrics.
- Calculate product category performance.
- Analyze order status metrics.
- Store analytical outputs as Delta datasets.

## Why Gold?

The Gold layer contains aggregated, business-oriented data
that is ready for analytics and reporting.

In [0]:
DATA_PATH = "/Volumes/workspace/default/olist_data"
SILVER_PATH = f"{DATA_PATH}/silver/order_details"

silver_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

print("Silver rows:", silver_df.count())

In [0]:
from pyspark.sql.functions import (
    countDistinct,
    sum,
    avg,
    round
)

daily_sales_df = (
    silver_df
    .groupBy("order_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_sales"),
        round(avg("price"), 2).alias("average_item_price")
    )
    .orderBy("order_date")
)

In [0]:
display(daily_sales_df)

In [0]:
category_sales_df = (
    silver_df
    .groupBy("product_category_name")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        countDistinct("product_id").alias("unique_products"),
        round(sum("price"), 2).alias("total_sales"),
        round(avg("price"), 2).alias("average_price")
    )
    .orderBy("total_sales", ascending=False)
)

In [0]:
display(category_sales_df)

In [0]:
order_status_df = (
    silver_df
    .groupBy("order_status")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_sales")
    )
    .orderBy("total_orders", ascending=False)
)

In [0]:
display(order_status_df)

In [0]:
GOLD_PATH = f"{DATA_PATH}/gold"

(
    daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/daily_sales")
)

(
    category_sales_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/category_sales")
)

(
    order_status_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{GOLD_PATH}/order_status")
)